In [ ]:
import os
import sys
import os
import pandas as pd
from PIL import Image

# Add project root to system path (for relative imports to work)
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

    
from src import config

In [ ]:
import os
import re
import csv
from pathlib import Path
from typing import Tuple, Dict, List
from PIL import Image
import numpy as np

def build_submission_from_labels_all(
    test_flat_dir: str,                   # images: test_flat/tomoId_sliceId.<ext>
    labels_dir: str,                      # labels: rtdetr_batch/exp/labels/tomoId_sliceId.txt
    out_csv_path: str = "submission.csv",
    header_style: str = "kaggle",         # "kaggle" or "simple"
    agg: str = "mean",                    # "mean" or "median"
    image_exts: Tuple[str, ...] = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".PNG", ".JPG", ".JPEG", ".TIF", ".TIFF", ".BMP"),
):
    labels_root = Path(labels_dir)
    test_flat = Path(test_flat_dir)

    # Index ALL images in test_flat by base name (stem without extension)
    # e.g., "tomo_0001_0012.png" -> key: "tomo_0001_0012"
    image_map: Dict[str, Path] = {}
    for p in test_flat.iterdir():
        if p.is_file() and p.suffix in image_exts:
            image_map[p.stem] = p

    # Helper: parse label file name to (tomo_id, slice_str)
    # Works even if tomo_id contains underscores: rsplit('_', 1)
    def parse_label_name(txt_path: Path):
        base = txt_path.stem
        if "_" not in base:
            return None
        tomo_id, slice_str = base.rsplit("_", 1)
        if not slice_str.isdigit():
            return None
        return tomo_id, slice_str

    # Scan label txts and collect detections per tomo
    all_label_files = sorted(labels_root.rglob("*.txt"))
    tomo_points: Dict[str, Dict[str, List[float]]] = {}  # tomo -> dict with xs, ys, zs
    missing_images: List[str] = []
    parsed_labels = 0

    for txt in all_label_files:
        parsed_labels += 1
        parsed = parse_label_name(txt)
        if parsed is None:
            # skip weirdly named file
            continue
        tomo_id, slice_str = parsed
        base = f"{tomo_id}_{slice_str}"

        # get image to recover W,H for de-normalization
        img_path = image_map.get(base)
        if img_path is None:
            missing_images.append(base)
            # cannot de-normalize without W,H -> skip this label
            continue

        # read image size
        try:
            with Image.open(img_path) as im:
                W, H = im.size
        except Exception:
            missing_images.append(base)
            continue

        # read all detections in this slice; use ALL (no conf filtering)
        try:
            with txt.open("r") as f:
                any_line = False
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 6:
                        continue
                    # cls conf x y w h  (normalized x,y)
                    try:
                        x_c = float(parts[2])
                        y_c = float(parts[3])
                    except ValueError:
                        continue
                    any_line = True
                    tomo_points.setdefault(tomo_id, {"xs": [], "ys": [], "zs": []})
                    tomo_points[tomo_id]["xs"].append(x_c * W)
                    tomo_points[tomo_id]["ys"].append(y_c * H)
                    tomo_points[tomo_id]["zs"].append(float(int(slice_str)))
                # empty file = no detection; just skip
        except Exception:
            # unreadable label file; skip
            continue

    # Build rows: one row per tomo_id (as evaluator requires)
    rows = []
    for tomo_id in sorted({tid for tid in tomo_points.keys()} | {parse_label_name(p)[0]
                                                                 for p in all_label_files
                                                                 if parse_label_name(p) is not None}):
        pts = tomo_points.get(tomo_id)
        if not pts or len(pts["xs"]) == 0:
            # No detections across slices for this tomo
            if header_style == "kaggle":
                rows.append({
                    "tomo_id": tomo_id,
                    "Motor axis 0": -1,
                    "Motor axis 1": -1,
                    "Motor axis 2": -1,
                    "Has motor": 0,
                })
            else:
                rows.append({
                    "tomoId": tomo_id, "x": -1, "y": -1, "z": -1, "Has Motor": 0
                })
        else:
            if agg == "median":
                x = float(np.median(pts["xs"]))
                y = float(np.median(pts["ys"]))
                z = float(np.median(pts["zs"]))
            else:
                x = float(np.mean(pts["xs"]))
                y = float(np.mean(pts["ys"]))
                z = float(np.mean(pts["zs"]))
            if header_style == "kaggle":
                rows.append({
                    "tomo_id": tomo_id,
                    "Motor axis 0": x,
                    "Motor axis 1": y,
                    "Motor axis 2": z,
                    "Has motor": 1,
                })
            else:
                rows.append({
                    "tomoId": tomo_id, "x": x, "y": y, "z": z, "Has Motor": 1
                })

    # Write CSV
    if header_style == "kaggle":
        fieldnames = ["tomo_id", "Motor axis 0", "Motor axis 1", "Motor axis 2", "Has motor"]
    else:
        fieldnames = ["tomoId", "x", "y", "z", "Has Motor"]

    Path(out_csv_path).parent.mkdir(parents=True, exist_ok=True)
    with open(out_csv_path, "w", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=fieldnames)
        wr.writeheader()
        for r in rows:
            wr.writerow(r)

    # Diagnostics
    total_labels = len(all_label_files)
    print(f"Labels scanned: {total_labels}")
    print(f"Labels parsed:  {parsed_labels}")
    print(f"Tomos in labels: {len(set(parse_label_name(p)[0] for p in all_label_files if parse_label_name(p) is not None))}")
    print(f"Tomos with detections: {sum(1 for r in rows if (r.get('Has motor', r.get('Has Motor')) == 1))}")
    if missing_images:
        print(f"WARNING: {len(missing_images)} label(s) skipped because matching image not found in test_flat.")
        print("Example missing base names:", missing_images[:10])
    print(f"Wrote {len(rows)} rows -> {out_csv_path} (agg='{agg}', header_style='{header_style}').")


In [ ]:
build_submission_from_labels_all(
    test_flat_dir="/data/horse/ws/kein254g-team_project/test_flat",
    labels_dir="/data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels",
    out_csv_path="./submission.csv",
    header_style="kaggle",   # or "simple" for (tomoId,x,y,z,Has Motor)
    agg="mean",             # or "median"s
)

In [ ]:
import glob
root_dir="/data/horse/ws/kein254g-team_project/rtdetr_batch/exp2"
labels_dir=f"{root_dir}/labels"

images_dir="/data/horse/ws/kein254g-team_project/test_flat"
txt_files = glob.glob(f"{labels_dir}/*.txt")

# /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2

rows = []
print(len(txt_files))

def get_image_width_height(image_path, file_name):
    parent_dir = os.path.dirname(labels_dir)
    file_name = os.path.splitext(file_name)[0] + ".jpg"
    #file_name = os.path.splitext(os.path.basename(txt_path))[0]   # tomo_01a877_slice_0148
    image_path = os.path.join(images_dir, f"{file_name}") 

    with Image.open(image_path) as img:
        return img.size  # returns (width, height)  

#for filename in glob.glob(f"{root_dir}/*.jpg"): 
#    im=Image.open(filename)
#    W,H = im.size
#    print(f"Image: {filename}, Width: {W}, Height: {H}")

for file_path in txt_files:
    print(f"\n--- Reading: {file_path} ---")
    with open(file_path, "r", encoding="utf-8") as f:
            file_name = os.path.basename(file_path)            # e.g. "image1.txt"
            file_id = os.path.splitext(file_name)[0]           # e.g. "image1"
            _,id,_,z = file_id.split('_')
            z = int(z)
            cls,confidence,x_center,y_center,width,height = f.readline().strip().split()
            id = "tomo_" + str(id)
            #print(f"cls :{cls}, confidence: {confidence}, x_center: {x_center}, y_center: {y_center}, width: {width}, height: {height}")
            #print(f"file_id: {file_id} , file_name: {file_name}")
            #print(f"id: {id}, z: {z}")

            #tomo_id,Motor axis 0,Motor axis 1,Motor axis 2,Has motor
            #tomo_id - unique identifier of the tomogram. Some tomograms in the train set have multiple motors.
            #Motor axis 0 - the z-coordinate of the motor, i.e., which slice it is located on
            #Motor axis 1 - the y-coordinate of the motor
            #Motor axis 2 - the x-coordinate of the motor

            #print(f"filename: {file_name}, file path: {file_path}, file id: {file_id} ")
            W,H = get_image_width_height(images_dir, file_name)
            print(f"Tomo Id: {id} Image Width: {W}, Height: {H}")

            #denomalize
            x = float(x_center) * W
            y = float(y_center) * H
            

            rows.append({
                    "tomo_id": id,               # just the name without extension
                    "Motor axis 0": z,            # full .txt file name
                    "Motor axis 1": y,
                    "Motor axis 2": x,
                })
            


df = pd.DataFrame(rows)
df.to_csv("./my_submisson.csv", index=False)
print("CSV created: my_submisson.csv")

    
    

46

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_00e047_slice_0170.txt ---
Tomo Id: 00e047 Image Width: 928, Height: 959

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_01a877_slice_0151.txt ---
Tomo Id: 01a877 Image Width: 928, Height: 960

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_01a877_slice_0139.txt ---
Tomo Id: 01a877 Image Width: 928, Height: 960

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_01a877_slice_0146.txt ---
Tomo Id: 01a877 Image Width: 928, Height: 960

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_00e047_slice_0176.txt ---
Tomo Id: 00e047 Image Width: 928, Height: 959

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_01a877_slice_0153.txt ---
Tomo Id: 01a877 Image Width: 928, Height: 960

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tom

In [ ]:
make_kaggle_submission_from_labels(
    labels_dir="/data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels",
    images_dir="/data/horse/ws/kein254g-team_project/test_flat",
    out_csv_path="./submission.csv",
)